# IOAI — 2024 Mock Competition Basketball Shooting — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/jaredliw/ioai-tsp-2025"
CLONE = "ioai-tsp-2025"
SUBDIR = "noai-china-2024/basketball-shooting"
WORKDIR = "noai-china-2024/basketball-shooting"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pickle

data = pd.read_csv('data_train.csv') #Load training set, please do not change this address.
data = data[~data.shot_made_flag.isna()] 
# Count the number of rows and columns. 
num_rows, num_columns = data.shape
column_names = data.columns
#print(f"Row：{num_rows}")
#print(f"Column：{num_columns}")
print("\n")
#print("Column Name:")
for col in column_names:
    print(col)


X_train = data[['loc_x', 'loc_y']].values.astype(np.float32) #Load features from training set 
y_train = data['shot_made_flag'].values.astype(np.float32) #Load label from training set 
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)


# Define the MLP model
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        #######Please design your model here########
        self.register_buffer('mean',torch.zeros(2))
        self.register_buffer('std',torch.ones(2))
        self.hidden1 = nn.Linear(2, 8)
        self.ac1 = nn.ELU()
        self.hidden2 = nn.Linear(8, 8)
        self.ac2 = nn.ELU()
        self.output = nn.Linear(8, 1)
        self.ac3 = nn.Sigmoid()
        
    def forward(self, x):
        #######Please design your model here########
        x=(x-self.mean)/self.std
        x = self.ac1(self.hidden1(x))
        x = self.ac2(self.hidden2(x))
        x = self.ac3(self.output(x))
        return x


def train(model, optimizer, criterion, X_train_tensor, y_train_tensor):
    train_losses = []
    val_losses = []
    num_epochs = 5000
    for epoch in range(num_epochs):
        # Training 
        model.train()
        
        #Forward calculation
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        
        #Backward training
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        #Record the training loss 
        train_losses.append(loss.item())
        
        # Print the training loss
        model.eval()
        if (epoch+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        scheduler.step()
    return train_losses   


model=MyModel()
# 입력 정규화 통계를 학습데이터로 채워 버퍼에 저장(모델과 함께 저장돼 채점 시에도 동일 정규화)
model.mean.copy_(torch.tensor(X_train.mean(axis=0)))
model.std.copy_(torch.tensor(X_train.std(axis=0)) + 1e-6)
# Define the loss function and the optimizer
criterion = nn.BCELoss()  # BCE Loss
optimizer = optim.AdamW(model.parameters(), lr=0.01)   # Adam optrimizer
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=5000)
train_losses = train(model, optimizer, criterion, X_train_tensor, y_train_tensor)

In [ ]:
# submission_model.py 의 MyModel 은 학습된 모델과 동일해야 채점 시 state_dict 로딩이 됩니다
# (mean/std 버퍼 + 정규화 포함).
model_code = """
import torch
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        #######Please design your model here########
        self.register_buffer('mean', torch.zeros(2))
        self.register_buffer('std', torch.ones(2))
        self.hidden1 = nn.Linear(2, 8)
        self.ac1 = nn.ELU()
        self.hidden2 = nn.Linear(8, 8)
        self.ac2 = nn.ELU()
        self.output = nn.Linear(8, 1)
        self.ac3 = nn.Sigmoid()

    def forward(self, x):
        #######Please design your model here########
        x = (x - self.mean) / self.std
        x = self.ac1(self.hidden1(x))
        x = self.ac2(self.hidden2(x))
        x = self.ac3(self.output(x))
        return x
"""
with open('submission_model.py', 'w') as f:
    f.write(model_code)
print("submission_model.py file is generated.")

In [ ]:
# Save the model parameter
torch.save(model.state_dict(), 'submission_dic.pth')

In [ ]:
import zipfile
import os

# Define the files to be packaged and the compressed file name. 
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Create a zip file to submit.
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add files to the zip file
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')

In [ ]:
# (Bohrium 전용 '!cp submission.zip /personal' 은 이 환경에선 불필요 — 
#  submission.zip 이 작업 폴더에 생성되어 Submissions 탭이 자동 수집합니다.)

When a bug occurs during direct submission from Bohrium’s file, you can download it to your local machine and then submit it from the local machine.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)